# MMVC Trainer - 3. Model Training

このノートブックでVITSモデルの学習を実行します。

## 1. 学習前の確認

In [ ]:
import os
import json
import torch
import matplotlib.pyplot as plt

# GPU環境の確認
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# データセットの確認
required_files = ['filelists/train.txt', 'filelists/val.txt']
for file_path in required_files:
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        print(f"✓ {file_path}: {len(lines)} samples")
    else:
        print(f"✗ {file_path}: Not found!")
        print("Please run '2. Create_Configfile.ipynb' first.")

In [ ]:
# 設定ファイルの選択
import glob

config_files = glob.glob('configs/*.json')
print("Available configuration files:")
for i, config_file in enumerate(config_files):
    print(f"  {i}: {config_file}")

if config_files:
    # 多話者設定があれば優先
    if 'configs/multispeaker_config.json' in config_files:
        selected_config = 'configs/multispeaker_config.json'
        print(f"\nSelected: {selected_config} (multi-speaker)")
    else:
        selected_config = 'configs/baseconfig.json'
        print(f"\nSelected: {selected_config} (single-speaker)")
        
    # 設定内容の表示
    with open(selected_config, 'r', encoding='utf-8') as f:
        config = json.load(f)
    
    print(f"\nConfiguration summary:")
    print(f"  - Speakers: {config['data'].get('n_speakers', 0)}")
    print(f"  - Batch size: {config['train']['batch_size']}")
    print(f"  - Learning rate: {config['train']['learning_rate']}")
    print(f"  - Epochs: {config['train']['epochs']}")
    print(f"  - Sampling rate: {config['data']['sampling_rate']}")

## 2. 学習設定の調整

In [ ]:
# GPU使用量に基づく設定調整
def adjust_config_for_gpu(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = json.load(f)
    
    # GPU メモリに基づくバッチサイズ調整
    if torch.cuda.is_available():
        gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Memory: {gpu_memory_gb:.1f} GB")
        
        if gpu_memory_gb < 8:
            # 低メモリ設定
            config['train']['batch_size'] = 4
            config['train']['segment_size'] = 4096
            print("Low memory GPU detected - using reduced settings")
        elif gpu_memory_gb < 16:
            # 中メモリ設定
            config['train']['batch_size'] = 8
            config['train']['segment_size'] = 6144
            print("Medium memory GPU detected - using moderate settings")
        else:
            # 高メモリ設定
            config['train']['batch_size'] = 16
            config['train']['segment_size'] = 8192
            print("High memory GPU detected - using full settings")
    
    # Colab向け設定
    config['train']['fp16_run'] = True  # メモリ節約
    config['train']['log_interval'] = 100  # ログ頻度を上げる
    config['train']['eval_interval'] = 500  # 評価頻度を上げる
    
    # 調整後の設定を保存
    adjusted_config_path = config_path.replace('.json', '_adjusted.json')
    with open(adjusted_config_path, 'w', encoding='utf-8') as f:
        json.dump(config, f, indent=2, ensure_ascii=False)
    
    print(f"Adjusted configuration saved to: {adjusted_config_path}")
    return adjusted_config_path

adjusted_config = adjust_config_for_gpu(selected_config)

## 3. 学習の開始

In [ ]:
# 学習ディレクトリの準備
import shutil
from datetime import datetime

# タイムスタンプ付きのモデルディレクトリ
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_dir = f"logs/mmvc_{timestamp}"
os.makedirs(model_dir, exist_ok=True)

# 設定ファイルをコピー
shutil.copy(adjusted_config, os.path.join(model_dir, 'config.json'))

print(f"Training directory: {model_dir}")
print(f"Config copied to: {model_dir}/config.json")

In [ ]:
# 学習スクリプトの選択と実行
with open(adjusted_config, 'r', encoding='utf-8') as f:
    config = json.load(f)

n_speakers = config['data'].get('n_speakers', 0)

if n_speakers > 1:
    train_script = 'train_ms.py'
    print(f"Starting multi-speaker training with {n_speakers} speakers...")
else:
    train_script = 'train.py'
    print("Starting single-speaker training...")

print(f"Using script: {train_script}")
print(f"Model directory: {model_dir}")
print(f"Configuration: {adjusted_config}")

In [ ]:
# 環境変数の設定
import os

# CUDA設定
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTHONPATH'] = '/content/MMVC_Trainer'

# モデルディレクトリを設定に反映
import json
with open(adjusted_config, 'r', encoding='utf-8') as f:
    config = json.load(f)

config['model_dir'] = model_dir
config['train']['eval_interval'] = 500  # 短めに設定

with open(adjusted_config, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("Environment configured for training")

In [ ]:
# 学習の実行（バックグラウンド）
import subprocess
import threading
import time

def run_training():
    """学習を実行"""
    cmd = [
        'python', train_script,
        '-c', adjusted_config,
        '-m', model_dir
    ]
    
    print(f"Starting training with command: {' '.join(cmd)}")
    
    try:
        # プロセスを開始
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True,
            bufsize=1
        )
        
        # リアルタイムでログを表示
        for line in iter(process.stdout.readline, ''):
            print(line.rstrip())
            
        process.wait()
        print(f"Training completed with exit code: {process.returncode}")
        
    except KeyboardInterrupt:
        print("Training interrupted by user")
        process.terminate()
    except Exception as e:
        print(f"Training failed: {e}")

# 学習開始の確認
start_training = input("Start training? (y/n): ")
if start_training.lower() == 'y':
    print("Starting training...")
    print("Note: Training may take several hours. You can interrupt with Ctrl+C.")
    run_training()
else:
    print("Training not started. You can run it manually with:")
    print(f"python {train_script} -c {adjusted_config} -m {model_dir}")

## 4. 学習の監視（別セルで実行）

In [ ]:
# TensorBoardでの学習監視
import subprocess

# TensorBoardの起動
def start_tensorboard(log_dir):
    try:
        # Colabの場合
        get_ipython().system_raw(
            'tensorboard --logdir {} --host 0.0.0.0 --port 6006 &'
            .format(log_dir)
        )
        print(f"TensorBoard started for {log_dir}")
        print("Access via: http://localhost:6006")
    except:
        print(f"Run manually: tensorboard --logdir {log_dir}")

start_tensorboard(model_dir)

In [ ]:
# 学習進捗の確認
import glob
import matplotlib.pyplot as plt

def check_training_progress(model_dir):
    """学習の進捗を確認"""
    # チェックポイントファイルの確認
    checkpoints = glob.glob(os.path.join(model_dir, "G_*.pth"))
    if checkpoints:
        latest_checkpoint = max(checkpoints, key=os.path.getctime)
        step = os.path.basename(latest_checkpoint).split('_')[1].split('.')[0]
        print(f"Latest checkpoint: {latest_checkpoint}")
        print(f"Training step: {step}")
    else:
        print("No checkpoints found yet")
    
    # ログファイルの確認
    log_files = glob.glob(os.path.join(model_dir, "*.log"))
    if log_files:
        latest_log = max(log_files, key=os.path.getctime)
        print(f"\nLatest log: {latest_log}")
        
        # 最新のログ行を表示
        with open(latest_log, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if lines:
                print("Recent log entries:")
                for line in lines[-5:]:
                    print(f"  {line.strip()}")

# 5秒ごとに進捗を確認
check_training_progress(model_dir)

## 学習完了後の次のステップ

学習が完了したら:
1. "4. MMVC_Interface.ipynb"で音声変換を試す
2. "5. Export_ONNX.ipynb"でモデルをONNXエクスポート

### 学習時間の目安
- 数分のデータ: 1-2時間
- 10分程度のデータ: 3-5時間
- 30分以上のデータ: 8-12時間

### トラブルシューティング
- GPU メモリ不足: バッチサイズを減らす
- 学習が進まない: 学習率を調整
- 音質が悪い: データの品質を確認